In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import os
from collections.abc import Iterable
from typing import Any

import pandas as pd
import torch
from hydra import compose, initialize
from hydra.core.global_hydra import GlobalHydra  # Import GlobalHydra explicitly
from hydra.utils import instantiate

from ogbench.utils.config_resolvers import (
    get_default_transform,
    get_monitor_metric,
    get_monitor_mode,
    infer_in_channels,
)

# stay off GPU
os.environ["CUDA_VISIBLE_DEVICES"] = ""
torch.set_grad_enabled(False)

PROJECT_CFG_NAME = "train.yaml"

# -------------------------
# Grids (matching your CLI)
# -------------------------
DATASETS = ["covidaki", "motrpac", "addneuromed", "parkinsons"]
ADJ_THRESHOLDS = [0.8, 0.85]  # from your example; add more if you want
DATALOADER_BATCH_SIZES = [8, 16]
NODE_SAMPLE_RATIOS = [1.0, 0.5, 0.2, 0.125, "full"]
SAMPLE_METHODS = ["variance", "random", "correlation"]

OPT_LRS = [0.001]
OPT_WD = [0.0004]  # your working config uses 0.0004

FE_OUT = [64, 128, 256]  # adjust if you want [32, 64, 128] like the example
# FE_PROJ_DROPOUT = [0.0, 0.25]

BB_NUM_LAYERS = [2, 4]
BB_DROPOUT = [0.2, 0.4]  # skip for chebnet
BB_ACT = ["relu"]  # skip for chebnet

READOUT_POOL = ["mean", "sum"]

# Models (use the exact names from your CLI)
MODEL_KEYS = [
    "sagn",
    "chebnet",
    "mlp",
    "gin",
    "gatv4",
    "gcn",
    "gat",
    "gatv2",
    "graph_sage",
]

# Model-specific grids (only things you actually have)
PER_MODEL_GRID: dict[str, dict[str, Iterable[Any]]] = {
    "gcn": {
        "model.backbone.num_layers": [2, 4],
    },
    "gat": {
        "model.backbone.heads": [2, 4],
        "model.backbone.num_layers": [2, 4],
        "model.backbone.v2": [False],
        # add if you truly have them:
        # "model.backbone.concat": [True],
    },
    "gin": {
        "model.backbone.num_layers": [2, 4],
    },
    "gatv2": {
        "model.backbone.v2": [True],
        "model.backbone.heads": [2, 4],
        "model.backbone.num_layers": [2, 4],
    },
    "gatv4": {
        "model.backbone.hidden_channels": [[8, 16], [64, 128]],
        "model.backbone.heads": [[3, 3]],
    },
    "graph_sage": {
        "model.backbone.num_layers": [2, 4],
    },
    "chebnet": {
        "model.backbone.K": [2, 3],
        "model.backbone.num_layers": [2, 4],
    },
    "mlp": {
        "model.backbone.hidden_channels": [
            [128, 64, 32],
            [512, 256, 128],
            [1024, 512, 256],
        ],
    },
    "sagn": {
        "model.backbone.hidden_channels": [128, 256],
        "model.backbone.dropout": [0.2, 0.4],
        "model.backbone.num_layers": [2, 4],
        "model.backbone.alpha": [0.5, 0.8],
    },
}

# -------------------------
# Helpers
# -------------------------


def count_trainable_params(model: torch.nn.Module) -> int:
    return sum(int(p.numel()) for p in model.parameters() if p.requires_grad)


def product_dict(grid: dict[str, Iterable[Any]]) -> list[dict[str, Any]]:
    keys = list(grid.keys())
    vals = [list(v) for v in grid.values()]
    return [dict(zip(keys, tup, strict=True)) for tup in itertools.product(*vals)]


def to_override(k: str, v: Any) -> str:
    if isinstance(v, bool):
        return f"{k}={'true' if v else 'false'}"
    if isinstance(v, str):
        return f"{k}={v}"
    if isinstance(v, (list, tuple)):
        inner = ",".join(str(x) for x in v)
        return f"{k}=[{inner}]"
    return f"{k}={v}"


def build_overrides(base_overrides: list[str], hp_dict: dict[str, Any]) -> list[str]:
    return base_overrides + [to_override(k, v) for k, v in hp_dict.items()]


# shared grid (keys exactly as in your CLI)
SHARED_GRID_BASE: dict[str, Iterable[Any]] = {
    # "dataset": DATASETS,
    # "dataset.loader.parameters.adjacency_threshold": ADJ_THRESHOLDS,
    # "dataset.dataloader_params.batch_size": DATALOADER_BATCH_SIZES,
    # "dataset.loader.parameters.node_sample_ratio": NODE_SAMPLE_RATIOS,
    # "dataset.loader.parameters.method": SAMPLE_METHODS,
    "optimizer.parameters.lr": OPT_LRS,
    "optimizer.parameters.weight_decay": OPT_WD,
    "model.feature_encoder.out_channels": FE_OUT,
    # "model.feature_encoder.proj_dropout": FE_PROJ_DROPOUT,
    "model.readout.pooling_type": READOUT_POOL,
}

GLOBAL_BACKBONE = {
    # "model.backbone.dropout": BB_DROPOUT,
    "model.backbone.act": BB_ACT,
}


def merged_shared_grid_for_model(model_key: str) -> dict[str, Iterable[Any]]:
    """
    Shared knobs (optimizer, encoder, readout) +
    selective backbone defaults (skip some for chebnet).
    """
    g = dict(SHARED_GRID_BASE)

    # --- prune special cases ---
    if model_key == "sagn":
        g.pop("model.feature_encoder.out_channels", None)
    # Only add global backbone if the model isn't chebnet
    if model_key != "chebnet" and model_key != "sagn":
        g.update(GLOBAL_BACKBONE)

    # Always add the model-specific grid
    g.update(PER_MODEL_GRID.get(model_key, {}))
    if model_key == "gatv4" or model_key == "sagn" or model_key == "mlp":
        g.pop("model.feature_encoder.out_channels", None)
    return g


# -------------------------
# Sweep + count
# -------------------------
records = []
out_dir = "./stats/param_counts"
os.makedirs(out_dir, exist_ok=True)

# Clear GlobalHydra instance if already initialized
if GlobalHydra().is_initialized():
    GlobalHydra().clear()

initialize(config_path="../configs", job_name="job")

for model_key in MODEL_KEYS:
    shared_grid = merged_shared_grid_for_model(model_key)
    model_specific = PER_MODEL_GRID.get(model_key, {})
    full_grid = dict(shared_grid)
    full_grid.update(model_specific)

    for hp in product_dict(full_grid):
        overrides = [f"model={model_key}"]
        overrides = build_overrides(overrides, hp)

        try:
            cfg = compose(
                config_name=PROJECT_CFG_NAME,
                overrides=overrides,
                return_hydra_config=True,
            )
            model = instantiate(
                cfg.model, evaluator=cfg.evaluator, optimizer=cfg.optimizer, loss=cfg.loss
            ).cpu()

            n_params = count_trainable_params(model)

            records.append(
                {
                    "model": model_key,
                    "params": n_params,
                    "overrides": " ".join(overrides),
                    **hp,
                }
            )
        except Exception as e:
            records.append(
                {
                    "model": model_key,
                    "params": None,
                    "overrides": " ".join(overrides),
                    "error": str(e),
                    **hp,
                }
            )

df = pd.DataFrame(records)
os.makedirs(out_dir, exist_ok=True)
all_csv = os.path.join(out_dir, "params_all_combos.csv")
df.to_csv(all_csv, index=False)


def pick_row(group: pd.DataFrame, which: str):
    sub = group.dropna(subset=["params"])
    if sub.empty:
        return {"params": None, "overrides": None}
    idx = sub["params"].idxmin() if which == "min" else sub["params"].idxmax()
    row = sub.loc[idx]
    return {"params": int(row["params"]), "overrides": row["overrides"]}


summary_rows = []
for m, g in df.groupby("model", dropna=False):
    mn = pick_row(g, "min")
    mx = pick_row(g, "max")
    summary_rows.append(
        {
            "model": m,
            "min_params": mn["params"],
            "min_overrides": mn["overrides"],
            "max_params": mx["params"],
            "max_overrides": mx["overrides"],
            "num_valid": int(g["params"].notna().sum()),
            "num_total": int(len(g)),
            "num_errors": int(g["params"].isna().sum()),
        }
    )

summary = pd.DataFrame(summary_rows)
summary_csv = os.path.join(out_dir, "params_summary.csv")
summary.to_csv(summary_csv, index=False)

print(summary)
print(f"Saved:\n- {all_csv}\n- {summary_csv}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import os
import glob

# Find and clear matplotlib font cache to detect newly installed CMU Serif
cache_locations = [
    os.path.join(os.path.expanduser('~'), '.matplotlib', 'fontlist*.json'),
    os.path.join(os.path.expanduser('~'), '.cache', 'matplotlib', 'fontlist*.json'),
]

for pattern in cache_locations:
    for cache_file in glob.glob(pattern):
        try:
            os.remove(cache_file)
            print(f"Cleared font cache: {cache_file}")
        except:
            pass

# Add CMU Serif font directly from system location
cmu_font_path = '/usr/share/fonts/truetype/cmu/cmunrm.ttf'
if os.path.exists(cmu_font_path):
    font_manager.fontManager.addfont(cmu_font_path)
    print("Added CMU Serif font directly")
else:
    print(f"Warning: CMU Serif font not found at {cmu_font_path}")

# Verify CMU Serif is available
available_fonts = [f.name for f in font_manager.fontManager.ttflist]
cmu_fonts = [f for f in available_fonts if 'CMU' in f or 'Computer Modern' in f]
if cmu_fonts:
    print(f"Found CMU fonts: {set(cmu_fonts)}")
else:
    print("Warning: CMU Serif not found. You may need to restart the Jupyter kernel.")

# Set font to CMU Serif (no fallbacks)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['CMU Serif']

# Increase all font sizes by 6
plt.rcParams['font.size'] = 18  # base font size (default is 10)
plt.rcParams['axes.titlesize'] = 22  # title size (default is 12)
plt.rcParams['axes.labelsize'] = 24  # label size (default is 10, increased by 12 for xlabel)
plt.rcParams['xtick.labelsize'] = 18  # x-axis tick labels (default is 10)
plt.rcParams['ytick.labelsize'] = 18  # y-axis tick labels (default is 10)
plt.rcParams['legend.fontsize'] = 18  # legend (default is 10)

# High quality output settings
plt.rcParams['figure.dpi'] = 300  # High resolution for display
plt.rcParams['savefig.dpi'] = 300  # High resolution for saved figures
plt.rcParams['savefig.format'] = 'pdf'  # PDF format
plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts in PDF (better quality)

# n = #graphs, p = avg #nodes per graph
data = [
    # ------------------------
    # SAT Solving (from your table)
    # ------------------------
    {"dataset": "SAT-S (VG)",  "n_graphs": 69596, "avg_nodes": 1323.96}, #
    {"dataset": "SAT-M (VCG)", "n_graphs": 69596, "avg_nodes": 8304.22}, #
    {"dataset": "SAT-L (LCG)", "n_graphs": 69596, "avg_nodes": 9628.12}, #

    # ------------------------
    # OGB (from your screenshot)
    # ------------------------
    {"dataset": "ogbg-molhiv",  "n_graphs": 41127,  "avg_nodes": 25.5},
    {"dataset": "ogbg-molpcba", "n_graphs": 437929, "avg_nodes": 26.0},
    {"dataset": "ogbg-ppa",     "n_graphs": 158100, "avg_nodes": 243.4},

    # ------------------------
    # TUDataset stats (chrsmrrs.github.io)
    # ------------------------
    {"dataset": "AIDS",             "n_graphs": 2000,  "avg_nodes": 15.69}, #
    {"dataset": "DD",               "n_graphs": 1178,  "avg_nodes": 284.32}, #
    {"dataset": "ENZYMES",          "n_graphs": 600,   "avg_nodes": 32.63}, #
    {"dataset": "PROTEINS",         "n_graphs": 1113,  "avg_nodes": 39.06}, #
    #{"dataset": "PROTEINS_full",    "n_graphs": 1113,  "avg_nodes": 39.06},
    {"dataset": "COLLAB",           "n_graphs": 5000,  "avg_nodes": 74.49}, #
    {"dataset": "IMDB-BINARY",      "n_graphs": 1000,  "avg_nodes": 19.77}, #
    {"dataset": "IMDB-MULTI",       "n_graphs": 1500,  "avg_nodes": 13.00}, #
    {"dataset": "REDDIT-BINARY",    "n_graphs": 2000,  "avg_nodes": 429.63}, #
    {"dataset": "REDDIT-MULTI-5K",  "n_graphs": 4999,  "avg_nodes": 508.52}, #
    {"dataset": "REDDIT-MULTI-12K", "n_graphs": 11929, "avg_nodes": 391.41}, #
    {"dataset": "FRANKENSTEIN",     "n_graphs": 4337,  "avg_nodes": 16.90}, #
    {"dataset": "Mutagenicity",     "n_graphs": 4337,  "avg_nodes": 30.32}, #
    {"dataset": "MUTAG",            "n_graphs": 188,   "avg_nodes": 17.93}, #
    {"dataset": "NCI1",             "n_graphs": 4110,  "avg_nodes": 29.87}, #
    {"dataset": "NCI109",           "n_graphs": 4127,  "avg_nodes": 29.68}, #
    {"dataset": "COX2",             "n_graphs": 467,  "avg_nodes": 41.22}, #
    {"dataset": "DHFR",             "n_graphs": 756,  "avg_nodes": 42.43}, #
    {"dataset": "BZR",              "n_graphs": 405,   "avg_nodes": 35.75}, #
    {"dataset": "PTC_MR",           "n_graphs": 344,   "avg_nodes": 14.29}, #



    # ------------------------
    # LRGB (PyG docs): Peptides-func
    # ------------------------
    {"dataset": "Peptides-func",    "n_graphs": 15535, "avg_nodes": 150.94}, #

    # ------------------------
    # MNIST Superpixels (PyG docs): 70k graphs, 75 nodes each
    # ------------------------
    {"dataset": "MNIST (superpixels)", "n_graphs": 70000, "avg_nodes": 75.0},
]

df = pd.DataFrame(data)

# 1) n/p bar plot (like your earlier figure concept)
df["n_over_p"] = df["n_graphs"] / df["avg_nodes"]
df_bar = df.sort_values("n_over_p", ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(df_bar["dataset"], df_bar["n_over_p"], color="#662D91")
ax.set_xscale("log")
ax.axvline(1.0, linestyle="--", linewidth=2, color="#FFDE17")
ax.set_title("Dataset regime: ratio n/p (graphs per avg #nodes)")
ax.set_xlabel("n / p (log scale)", fontsize=20)  # Larger xlabel
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("output_barplot_classification.pdf", bbox_inches='tight', dpi=300)
plt.show()

# 2) log-log scatter of (p, n)
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df["avg_nodes"], df["n_graphs"])
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Avg #nodes per graph (p)")
ax.set_ylabel("#graphs (n)")
ax.set_title("Datasets in (p, n) space")

# label points
for _, r in df.iterrows():
    ax.annotate(r["dataset"], (r["avg_nodes"], r["n_graphs"]), fontsize=18)

plt.tight_layout()
plt.show()